# 05 ? Flat versus Shared-Hard comparison

**Objective:** Compute paired statistics, routing diagnostics and historical/new differences.

**Experiment:** Historical Flat versus Shared-Hard baseline reconstruction, seed 42.

**Config:** `configs/experiments/flat/efficientnet_b0.yaml`

**Inputs:** Hashed B0 Flat/Shared-Hard prediction CSVs and historical reference table.

**Outputs:** Paired bootstrap replicates, exact McNemar, comparison tables and figure.

**Mode:** Stored-prediction analysis only. Every input is loaded from disk; no other notebook's kernel state is required.

Use **Restart Kernel ? Run All**. Real training is disabled until the build handover is reviewed.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/protocol.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
from src.config import load_config
DATA_ROOT = os.environ.get("SKIN_CANCER_DATA_ROOT")
print("Dataset root:", Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else "NOT CONFIGURED ? set SKIN_CANCER_DATA_ROOT before image verification/training")


In [ ]:
config = load_config("configs/experiments/flat/efficientnet_b0.yaml")
import pandas as pd
display(pd.read_csv(ROOT/"results/historical/document_reported_internal_test.csv"))

## Paired analysis

10,000 paired ground-truth-stratified image bootstrap replicates, seed 42, linear percentile intervals. Direction is Shared-Hard minus Flat. Exact McNemar tests paired correctness, not macro-F1. Bootstrap uncertainty does not estimate training-seed variance.

In [ ]:
RUN_COMPARISON = False
from src.statistics import compare_evaluations
from src.reporting import plot_comparison
if RUN_COMPARISON:
    result, output_directory = compare_evaluations("flat_efficientnet_b0_seed42_test", "shared_hard_efficientnet_b0_seed42_test", config)
    display(result)
    display(pd.read_csv(output_directory/"historical_vs_reconstructed.csv"))
    display(plot_comparison(result, ROOT/"results/figures/efficientnet_b0_internal_test.png"))
    print("Artifacts:", output_directory)
else:
    print("Reconstructed paired comparison: NOT RUN")

## Reproduction review before remaining six pairs

Review implementation fidelity, validation learning curves, selected epochs, endpoint/oracle metrics and historical differences. No acceptance tolerance has been invented. Do not tune hyperparameters to match test scores. Only after researcher acceptance should the remaining six pairs be released.

In [ ]:
ACCEPT_REPRODUCTION = False
REVIEWER = ""
RATIONALE = ""
if ACCEPT_REPRODUCTION:
    from src.reporting import record_reproduction_review
    display(record_reproduction_review(config, reviewer=REVIEWER, rationale=RATIONALE, accept=True))
else:
    print("Remaining backbone training gate: NOT RELEASED")

## Summary and next step

Review the status and metrics displayed above. Missing artifacts mean **not run**, never a successful reproduction. Generated artifacts are listed in the output cells; scientific results remain separate from historical reference values.

**Next:** `Return to 02 and 03 with each remaining architecture only after reproduction review; future notebooks begin at 06`. Preserve completed run directories, prediction files and checkpoint backups before continuing.
